# NER + NEL Evaluation — NutriGraphRAG / Merlin

This notebook evaluates NER models and NEL variants independently and in combination.

## Structure
| Cell | What it does |
|------|--------------|
| 2 | Setup and imports — CUDA check |
| 3 | Evaluation passage + ground truth (38 entities) |
| 4 | NER utility functions (Jaccard, precision, recall) |
| 5 | GLiNER NER (`urchade/gliner_biomed-v0.1`) |
| 6 | spaCy NER (trf → lg → sm) |
| 7 | scispaCy NER (`en_ner_bc5cdr_md`) |
| 8 | NER summary table |
| 9 | NEL prerequisites check |
| 10 | NEL utility functions |
| 11 | Variant A — Lexical |
| 12 | Variant B — HNSW + SapBERT |
| 13 | Variant B — HNSW + BioLORD |
| 14 | Variant B — HNSW + MiniLM |
| 15 | Variant B — HNSW + MPNET |
| 16 | Variant C — HNSW + FoodSEM (GPU) |
| 17 | Variant D — FoodSEM + label-aware ordering (Idea 1) |
| 18 | Variant D+2 — FoodSEM + robust URI parsing only (Idea 2) |
| 19 | Variant E — FoodSEM + label-aware ordering + robust URI parsing (Ideas 1+2) |

**FoodSEM cells (16–17) skip automatically if CUDA is not available.**

In [1]:
import os
import sys
import time

# ── Cache: use personal cache to avoid shared cache permission conflicts ───────
# Other users own lock files in /mnt/data/huggingface_cache — do not use it here.
# Llama is only in the shared cache — we override HF_HOME in Cells 16/17 only.
os.environ["HF_HOME"]           = "/home/spapadias/.cache/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/home/spapadias/.cache/huggingface"
os.environ["HF_DATASETS_CACHE"]  = "/home/spapadias/.cache/huggingface"

# ── Project root ──────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── CUDA ──────────────────────────────────────────────────────────────────────
import torch
CUDA_AVAILABLE = torch.cuda.is_available()
print(f"CUDA available: {CUDA_AVAILABLE}")
if CUDA_AVAILABLE:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")
else:
    print("No GPU — Variants C and D will be skipped.")

# ── HuggingFace auth ──────────────────────────────────────────────────────────
from huggingface_hub import login, get_token
try:
    token = get_token()
    if token:
        # Use classic token — fine-grained token lacks gated repo permissions
        classic_token = "<HF_TOKEN_REDACTED>"
        os.environ["HF_TOKEN"]               = classic_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = classic_token
        login(token=classic_token, add_to_git_credential=False)
        print(f"HF_TOKEN: {classic_token[:8]}... (classic token)")
    else:
        print("WARNING: No HuggingFace token. Run: huggingface-cli login")
except Exception as e:
    print(f"HF login warning: {e}")

print(f"HF_HOME: {os.environ['HF_HOME']}")

/home/spapadias/.conda/envs/merlin_cuda/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/home/spapadias/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: False
No GPU — Variants C and D will be skipped.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF_TOKEN: hf_uEnob... (classic token)
HF_HOME: /home/spapadias/.cache/huggingface


## Cell 3 — Evaluation Passage and Ground Truth

The passage is a dense nutrition/diet text covering foods, nutrients, diets, biomarkers, and populations.
Ground truth contains 38 entities spanning all these categories.
**What to look for:** NER recall vs. this GT set tells you how much of the entity space each model can see.

In [2]:
PASSAGE = """The Mediterranean diet, widely adopted in Greece, Italy, and Spain, emphasizes
daily consumption of olive oil, whole wheat bread, legumes, and vegetables such
as spinach and tomatoes. Fatty fish including salmon provides omega-3 fatty acids,
shown to lower LDL cholesterol and reduce cardiovascular risk. A cohort study in
Japan found that blueberries and almonds improved HbA1c control in type 2 diabetes
patients, particularly postmenopausal women. Adequate intake of vitamin D, calcium,
iron, and folate is essential for metabolic health. Iron deficiency anaemia remains
prevalent globally. The DASH diet restricts sodium while promoting dietary fiber,
potassium, and magnesium. Greek yogurt provides calcium and protein with lower
lactose content. Polyphenols in blueberries and dark chocolate act as antioxidants.
Carotenoids in carrots and sweet potatoes are precursors to vitamin A. Fish oil
capsules supplement omega-3 fatty acid intake. BMI is widely used as a proxy for
adiposity in clinical research."""

GROUND_TRUTH = {
    "Mediterranean diet", "DASH diet",
    "olive oil", "whole wheat bread", "spinach", "salmon", "blueberries",
    "almonds", "Greek yogurt", "dark chocolate", "carrots", "sweet potatoes",
    "fish oil capsules", "legumes", "tomatoes",
    "omega-3 fatty acids", "vitamin D", "calcium", "iron", "folate",
    "dietary fiber", "potassium", "magnesium", "vitamin A", "sodium",
    "protein", "lactose",
    "polyphenols", "carotenoids",
    "LDL cholesterol", "HbA1c", "BMI",
    "postmenopausal women", "type 2 diabetes patients",
    "Greece", "Italy", "Spain", "Japan",
}

print(f"Ground truth size: {len(GROUND_TRUTH)} entities")
print("\nGround truth entities:")
for e in sorted(GROUND_TRUTH):
    print(f"  {e}")

Ground truth size: 38 entities

Ground truth entities:
  BMI
  DASH diet
  Greece
  Greek yogurt
  HbA1c
  Italy
  Japan
  LDL cholesterol
  Mediterranean diet
  Spain
  almonds
  blueberries
  calcium
  carotenoids
  carrots
  dark chocolate
  dietary fiber
  fish oil capsules
  folate
  iron
  lactose
  legumes
  magnesium
  olive oil
  omega-3 fatty acids
  polyphenols
  postmenopausal women
  potassium
  protein
  salmon
  sodium
  spinach
  sweet potatoes
  tomatoes
  type 2 diabetes patients
  vitamin A
  vitamin D
  whole wheat bread


## Cell 4 — NER Utility Functions

Pure Python helpers for evaluating NER output against ground truth.
**Jaccard** is the primary metric (intersection over union). **GT Coverage** shows recall against the ground truth set.

In [3]:
def jaccard(set_a, set_b):
    set_a = {s.lower() for s in set_a}
    set_b = {s.lower() for s in set_b}
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)


def evaluate_ner(extracted, ground_truth):
    extracted_lower = {s.lower() for s in extracted}
    gt_lower = {s.lower() for s in ground_truth}
    tp = len(extracted_lower & gt_lower)
    fp = len(extracted_lower - gt_lower)
    fn = len(gt_lower - extracted_lower)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    j = jaccard(extracted_lower, gt_lower)
    return {
        "extracted": len(extracted_lower),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "jaccard": j,
        "tp": tp, "fp": fp, "fn": fn,
    }


def print_ner_result(name, metrics, runtime):
    print(f"\n{'='*60}")
    print(f"NER: {name}  ({runtime:.2f}s)")
    print(f"{'='*60}")
    print(f"  Extracted:  {metrics['extracted']}  (TP={metrics['tp']}, FP={metrics['fp']}, FN={metrics['fn']})")
    print(f"  Precision:  {metrics['precision']:.3f}")
    print(f"  Recall:     {metrics['recall']:.3f}")
    print(f"  F1:         {metrics['f1']:.3f}")
    print(f"  Jaccard:    {metrics['jaccard']:.3f}")


print("NER utility functions defined.")

NER utility functions defined.


## Cell 5a — GLiNER NER

Model: `urchade/gliner_biomed-v0.1` — zero-shot NER using food/nutrition labels.
**What to look for:** GLiNER is expected to have the highest recall on food/nutrient entities. Watch for FP on generic terms (countries, populations may or may not be desired).

In [4]:
from gliner import GLiNER

GLINER_LABELS = [
    "food", "nutrient", "micronutrient", "macronutrient",
    "food component", "dietary supplement", "dietary pattern",
    "medical condition", "biomarker",
    "Country", "Population", "Measurement",
]

t0 = time.time()
gliner_model = GLiNER.from_pretrained("urchade/gliner_large_bio-v0.1")
gliner_entities_raw = gliner_model.predict_entities(PASSAGE, GLINER_LABELS, threshold=0.4) # was 0.5
gliner_runtime = time.time() - t0

gliner_extracted = set(e["text"] for e in gliner_entities_raw)

print(f"GLiNER runtime: {gliner_runtime:.2f}s")
print(f"\nExtracted {len(gliner_extracted)} entities:")
for e in sorted(gliner_entities_raw, key=lambda x: x["label"]):
    print(f"  [{e['label']:20s}] {e['text']}  (score={e['score']:.3f})")


gliner_metrics = evaluate_ner(gliner_extracted, GROUND_TRUTH)
print_ner_result("GLiNER biomed-v0.1", gliner_metrics, gliner_runtime)

2026-05-17 21:48:54.515167472 [W:onnxruntime:Default, device_discovery.cc:283 GetGpuDevices] Failed to detect devices under "/sys/class/drm/card0": device_discovery.cc:93 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
/home/spapadias/.conda/envs/merlin_cuda/lib/python3.12/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/home/spapadias/.conda/envs/merlin_cuda/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 60133.39it/s]
/home/spapadias/.conda/envs/merlin_cuda/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The senten

GLiNER runtime: 7.05s

Extracted 40 entities:
  [Country             ] Greece  (score=0.996)
  [Country             ] Italy  (score=0.989)
  [Country             ] Spain  (score=0.993)
  [Country             ] Japan  (score=0.988)
  [Measurement         ] BMI  (score=0.982)
  [Population          ] type 2 diabetes
patients  (score=0.734)
  [Population          ] postmenopausal women  (score=0.974)
  [biomarker           ] LDL cholesterol  (score=0.678)
  [biomarker           ] HbA1c  (score=0.899)
  [dietary pattern     ] Mediterranean diet  (score=0.979)
  [dietary pattern     ] DASH diet  (score=0.978)
  [dietary supplement  ] Fish oil
capsules  (score=0.971)
  [food                ] olive oil  (score=0.920)
  [food                ] whole wheat bread  (score=0.908)
  [food                ] legumes  (score=0.876)
  [food                ] vegetables  (score=0.552)
  [food                ] spinach  (score=0.709)
  [food                ] tomatoes  (score=0.786)
  [food                ] F

In [5]:
# Cell 5b — GLiNER large v2.1 (general purpose, larger training set)
# Compare against the biomed variant to see which handles nutrition labels better.
# Also tests nested NER (flat_ner=False) which may improve compound entity detection.

from gliner import GLiNER

t0 = time.time()
gliner_v2_model = GLiNER.from_pretrained("urchade/gliner_large-v2.1")
gliner_v2_runtime_load = time.time() - t0
print(f"Load time: {gliner_v2_runtime_load:.2f}s")

# Test 1: standard flat NER (same as Cell 5)
t0 = time.time()
gliner_v2_entities_flat = gliner_v2_model.predict_entities(
    PASSAGE, GLINER_LABELS, threshold=0.5, flat_ner=True
)
gliner_v2_runtime_flat = time.time() - t0

gliner_v2_extracted_flat = set(e["text"] for e in gliner_v2_entities_flat)
print(f"\nGLiNER v2.1 large (flat_ner=True)  runtime: {gliner_v2_runtime_flat:.2f}s")
print(f"Extracted {len(gliner_v2_extracted_flat)} entities:")
for e in sorted(gliner_v2_entities_flat, key=lambda x: x["label"]):
    print(f"  [{e['label']:20s}] {e['text']}  (score={e['score']:.3f})")

gliner_v2_metrics_flat = evaluate_ner(gliner_v2_extracted_flat, GROUND_TRUTH)
print_ner_result("GLiNER large-v2.1 (flat)", gliner_v2_metrics_flat, gliner_v2_runtime_flat)

# Test 2: nested NER (flat_ner=False) — finds entities within entities
t0 = time.time()
gliner_v2_entities_nested = gliner_v2_model.predict_entities(
    PASSAGE, GLINER_LABELS, threshold=0.5, flat_ner=False
)
gliner_v2_runtime_nested = time.time() - t0

gliner_v2_extracted_nested = set(e["text"] for e in gliner_v2_entities_nested)
print(f"\nGLiNER v2.1 large (flat_ner=False, nested) runtime: {gliner_v2_runtime_nested:.2f}s")

# Show only entities that differ from flat NER
new_in_nested = gliner_v2_extracted_nested - gliner_v2_extracted_flat
lost_in_nested = gliner_v2_extracted_flat - gliner_v2_extracted_nested
print(f"New entities (only in nested): {sorted(new_in_nested)}")
print(f"Lost entities (only in flat):  {sorted(lost_in_nested)}")

gliner_v2_metrics_nested = evaluate_ner(gliner_v2_extracted_nested, GROUND_TRUTH)
print_ner_result("GLiNER large-v2.1 (nested)", gliner_v2_metrics_nested, gliner_v2_runtime_nested)

# Use flat as the canonical result for downstream comparison
gliner_v2_extracted = gliner_v2_extracted_flat
gliner_v2_metrics = gliner_v2_metrics_flat

Ignored error while writing commit hash to /mnt/data/huggingface_cache/models--urchade--gliner_large-v2.1/refs/main: [Errno 13] Permission denied: '/mnt/data/huggingface_cache/models--urchade--gliner_large-v2.1/refs/main'.
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 40233.13it/s]
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Load time: 6.34s

GLiNER v2.1 large (flat_ner=True)  runtime: 0.42s
Extracted 38 entities:
  [Country             ] Greece  (score=0.979)
  [Country             ] Italy  (score=0.974)
  [Country             ] Spain  (score=0.973)
  [Country             ] Japan  (score=0.988)
  [Measurement         ] BMI  (score=0.974)
  [Population          ] type 2 diabetes
patients  (score=0.888)
  [Population          ] postmenopausal women  (score=0.979)
  [biomarker           ] HbA1c  (score=0.853)
  [dietary pattern     ] The Mediterranean diet  (score=0.943)
  [dietary pattern     ] DASH diet  (score=0.954)
  [dietary supplement  ] Fish oil
capsules  (score=0.978)
  [food                ] olive oil  (score=0.939)
  [food                ] whole wheat bread  (score=0.927)
  [food                ] legumes  (score=0.968)
  [food                ] vegetables  (score=0.838)
  [food                ] spinach  (score=0.920)
  [food                ] tomatoes  (score=0.918)
  [food                ] Fatty fi

In [6]:
# Cell 5c — GLiNER Large Bio v0.1: Advanced Configuration Tests
# Tests three advanced configurations to see if we can improve on the
# baseline Jaccard=0.810 achieved in Cell 5 with threshold=0.5, flat_ner=True.
#
# Test 1: lower threshold (0.4) — recovers low-confidence entities
# Test 2: nested NER (flat_ner=False) — better span detection for compound terms
# Test 3: per-label thresholds — domain-aware confidence requirements

print("="*65)
print("GLiNER Large Bio v0.1 — Advanced Configuration Tests")
print("="*65)

# --- Test 1: Lower threshold (0.4) ---
print("\n--- Test 1: threshold=0.4 (vs baseline 0.5) ---")
t0 = time.time()
entities_t04 = gliner_model.predict_entities(
    PASSAGE, GLINER_LABELS, threshold=0.4
)
rt_t04 = time.time() - t0

extracted_t04 = set(e["text"] for e in entities_t04)
metrics_t04 = evaluate_ner(extracted_t04, GROUND_TRUTH)
print_ner_result("Bio v0.1 threshold=0.4", metrics_t04, rt_t04)

# Show what changed vs baseline
new_entities = extracted_t04 - gliner_extracted
lost_entities = gliner_extracted - extracted_t04
print(f"  New vs baseline (threshold=0.5): {sorted(new_entities)}")
print(f"  Lost vs baseline: {sorted(lost_entities)}")

# Show scores for new entities
new_with_scores = [(e["text"], e["score"], e["label"]) 
                   for e in entities_t04 
                   if e["text"] in new_entities]
print(f"  New entity scores: {sorted(new_with_scores, key=lambda x: x[1])}")

# --- Test 2: Nested NER ---
print("\n--- Test 2: flat_ner=False (nested NER) ---")
t0 = time.time()
entities_nested = gliner_model.predict_entities(
    PASSAGE, GLINER_LABELS, threshold=0.5, flat_ner=False
)
rt_nested = time.time() - t0

extracted_nested = set(e["text"] for e in entities_nested)
metrics_nested = evaluate_ner(extracted_nested, GROUND_TRUTH)
print_ner_result("Bio v0.1 nested NER", metrics_nested, rt_nested)

new_nested = extracted_nested - gliner_extracted
lost_nested = gliner_extracted - extracted_nested
print(f"  New vs baseline: {sorted(new_nested)}")
print(f"  Lost vs baseline: {sorted(lost_nested)}")

# --- Test 3: Per-label thresholds ---
print("\n--- Test 3: Per-label thresholds ---")
# Rationale:
# - Food items score high (0.7+) → threshold 0.5 is fine
# - Biomarkers/clinical terms score lower → allow 0.35
# - Micronutrients score lower → allow 0.4
# - Countries/populations score high → threshold 0.5 is fine
LABEL_THRESHOLDS = {
    "food": 0.50,
    "nutrient": 0.40,
    "micronutrient": 0.38,
    "macronutrient": 0.40,
    "food component": 0.40,
    "dietary supplement": 0.50,
    "dietary pattern": 0.50,
    "medical condition": 0.40,
    "biomarker": 0.35,       # LDL cholesterol scored 0.678 — lower floor for clinical terms
    "Country": 0.50,
    "Population": 0.45,
    "Measurement": 0.50,
}

t0 = time.time()
# Run with the lowest threshold to get all candidates
all_candidates = gliner_model.predict_entities(
    PASSAGE, GLINER_LABELS, threshold=0.30
)
rt_perlabel = time.time() - t0

# Filter per label
extracted_perlabel = set()
for e in all_candidates:
    label_threshold = LABEL_THRESHOLDS.get(e["label"], 0.50)
    if e["score"] >= label_threshold:
        extracted_perlabel.add(e["text"])

metrics_perlabel = evaluate_ner(extracted_perlabel, GROUND_TRUTH)
print_ner_result("Bio v0.1 per-label thresholds", metrics_perlabel, rt_perlabel)

new_perlabel = extracted_perlabel - gliner_extracted
lost_perlabel = gliner_extracted - extracted_perlabel
print(f"  New vs baseline: {sorted(new_perlabel)}")
print(f"  Lost vs baseline: {sorted(lost_perlabel)}")

# Show all candidates between 0.30 and 0.50 (the ones the per-label filter acts on)
borderline = [(e["text"], e["label"], e["score"]) 
              for e in all_candidates 
              if 0.30 <= e["score"] < 0.50]
print(f"\n  Borderline entities (0.30-0.50 score range):")
for text, label, score in sorted(borderline, key=lambda x: x[2], reverse=True):
    threshold = LABEL_THRESHOLDS.get(label, 0.50)
    accepted = "✓" if score >= threshold else "✗"
    print(f"    {accepted} [{label:20s}] {text} (score={score:.3f}, threshold={threshold})")

# --- Summary: which configuration wins? ---
print("\n" + "="*65)
print("CONFIGURATION COMPARISON — GLiNER Large Bio v0.1")
print("="*65)
configs = [
    ("Baseline (threshold=0.5, flat)", gliner_metrics),
    ("threshold=0.4, flat",            metrics_t04),
    ("threshold=0.5, nested",          metrics_nested),
    ("Per-label thresholds",           metrics_perlabel),
]
print(f"{'Configuration':<35} {'Precision':>9} {'Recall':>7} {'F1':>6} {'Jaccard':>8}")
print("-"*68)
for name, m in configs:
    print(f"{name:<35} {m['precision']:>9.3f} {m['recall']:>7.3f} "
          f"{m['f1']:>6.3f} {m['jaccard']:>8.3f}")
print("="*68)

# Identify best configuration by F1
best_config_name, best_config_metrics = max(configs, key=lambda x: x[1]['f1'])
print(f"\nBest configuration by F1: {best_config_name}")
print(f"  F1={best_config_metrics['f1']:.3f}, "
      f"Jaccard={best_config_metrics['jaccard']:.3f}")
print("\nUse this configuration in LinearRAGConfig for production indexing.")

GLiNER Large Bio v0.1 — Advanced Configuration Tests

--- Test 1: threshold=0.4 (vs baseline 0.5) ---

NER: Bio v0.1 threshold=0.4  (0.42s)
  Extracted:  39  (TP=35, FP=4, FN=3)
  Precision:  0.897
  Recall:     0.921
  F1:         0.909
  Jaccard:    0.833
  New vs baseline (threshold=0.5): []
  Lost vs baseline: []
  New entity scores: []

--- Test 2: flat_ner=False (nested NER) ---

NER: Bio v0.1 nested NER  (0.41s)
  Extracted:  39  (TP=34, FP=5, FN=4)
  Precision:  0.872
  Recall:     0.895
  F1:         0.883
  Jaccard:    0.791
  New vs baseline: ['Iron deficiency anaemia']
  Lost vs baseline: ['Polyphenols']

--- Test 3: Per-label thresholds ---

NER: Bio v0.1 per-label thresholds  (0.38s)
  Extracted:  39  (TP=35, FP=4, FN=3)
  Precision:  0.897
  Recall:     0.921
  F1:         0.909
  Jaccard:    0.833
  New vs baseline: []
  Lost vs baseline: []

  Borderline entities (0.30-0.50 score range):
    ✓ [micronutrient       ] Polyphenols (score=0.491, threshold=0.38)
    ✗ [Meas

## Cell 6 — spaCy NER

General-purpose NER. Tries trf → lg → sm in order.
**What to look for:** spaCy will catch GPE (countries) and ORG entities well but will miss most food/nutrition terms. Use as baseline.

In [7]:
import spacy

SKIP_LABELS = {"ORDINAL", "CARDINAL", "DATE", "TIME", "PERCENT", "MONEY", "QUANTITY"}

spacy_model_name = None
for candidate in ["en_core_web_trf", "en_core_web_lg", "en_core_web_sm"]:
    try:
        nlp = spacy.load(candidate)
        spacy_model_name = candidate
        break
    except OSError:
        continue

if spacy_model_name is None:
    print("No spaCy model found. Install with: python -m spacy download en_core_web_lg")
    spacy_extracted = set()
    spacy_metrics = evaluate_ner(set(), GROUND_TRUTH)
    spacy_runtime = 0.0
else:
    t0 = time.time()
    doc = nlp(PASSAGE)
    spacy_runtime = time.time() - t0
    spacy_extracted = {ent.text for ent in doc.ents if ent.label_ not in SKIP_LABELS}

    print(f"spaCy model: {spacy_model_name}  ({spacy_runtime:.2f}s)")
    print(f"\nExtracted {len(spacy_extracted)} entities (after filtering {SKIP_LABELS}):")
    for ent in doc.ents:
        if ent.label_ not in SKIP_LABELS:
            print(f"  [{ent.label_:15s}] {ent.text}")

    spacy_metrics = evaluate_ner(spacy_extracted, GROUND_TRUTH)
    print_ner_result(f"spaCy ({spacy_model_name})", spacy_metrics, spacy_runtime)

spaCy model: en_core_web_trf  (0.12s)

Extracted 5 entities (after filtering {'CARDINAL', 'DATE', 'MONEY', 'QUANTITY', 'ORDINAL', 'TIME', 'PERCENT'}):
  [LOC            ] Mediterranean
  [GPE            ] Greece
  [GPE            ] Italy
  [GPE            ] Spain
  [GPE            ] Japan

NER: spaCy (en_core_web_trf)  (0.12s)
  Extracted:  5  (TP=4, FP=1, FN=34)
  Precision:  0.800
  Recall:     0.105
  F1:         0.186
  Jaccard:    0.103


/home/spapadias/.local/lib/python3.12/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


## Cell 7 — scispaCy NER

Model: `en_ner_bc5cdr_md` — trained on BioCreative V CDR corpus (chemicals + diseases).
**What to look for:** Good on nutrients and clinical terms (HbA1c, LDL), but trained on disease/chemical pairs so coverage of food items varies.

In [8]:
# Cell 7 — scispaCy NER
# en_core_sci_scibert skipped: its DeBERTa backbone creates lock files in the
# shared cache owned by another user (vpitsilou), causing PermissionError.
# en_core_sci_lg is sufficient for the comparison — the conclusion (GLiNER wins)
# is not affected by whether we use scibert or lg here.

try:
    import spacy as _spacy

    for candidate in ["en_core_sci_lg", "en_core_sci_md", "en_ner_bc5cdr_md"]:
        try:
            nlp_sci = _spacy.load(candidate)
            sci_model_used = candidate
            break
        except OSError:
            continue
    else:
        raise OSError("No scispaCy model found")

    print(f"Using: {sci_model_used}")
    t0 = time.time()
    doc_sci = nlp_sci(PASSAGE)
    scispacy_runtime = time.time() - t0
    scispacy_extracted = {ent.text for ent in doc_sci.ents}

    print(f"Runtime: {scispacy_runtime:.2f}s | Extracted: {len(scispacy_extracted)}")
    for ent in doc_sci.ents:
        print(f"  [{ent.label_:15s}] {ent.text}")

    scispacy_metrics = evaluate_ner(scispacy_extracted, GROUND_TRUTH)
    print_ner_result(f"scispaCy ({sci_model_used})", scispacy_metrics, scispacy_runtime)

except OSError:
    print("No scispaCy model found. Install: pip install scispacy")
    scispacy_extracted = set()
    scispacy_metrics   = evaluate_ner(set(), GROUND_TRUTH)
    scispacy_runtime   = 0.0
except ImportError:
    print("scispaCy not installed.")
    scispacy_extracted = set()
    scispacy_metrics   = evaluate_ner(set(), GROUND_TRUTH)
    scispacy_runtime   = 0.0

Using: en_core_sci_lg
Runtime: 0.03s | Extracted: 62
  [ENTITY         ] Mediterranean
  [ENTITY         ] diet
  [ENTITY         ] Greece
  [ENTITY         ] Italy
  [ENTITY         ] Spain
  [ENTITY         ] daily
  [ENTITY         ] consumption
  [ENTITY         ] olive oil
  [ENTITY         ] wheat bread
  [ENTITY         ] legumes
  [ENTITY         ] vegetables
  [ENTITY         ] spinach
  [ENTITY         ] tomatoes
  [ENTITY         ] Fatty fish
  [ENTITY         ] salmon
  [ENTITY         ] omega-3 fatty acids
  [ENTITY         ] lower
  [ENTITY         ] LDL cholesterol
  [ENTITY         ] reduce
  [ENTITY         ] cardiovascular risk
  [ENTITY         ] cohort study
  [ENTITY         ] Japan
  [ENTITY         ] blueberries
  [ENTITY         ] almonds
  [ENTITY         ] HbA1c
  [ENTITY         ] control
  [ENTITY         ] type 2 diabetes
  [ENTITY         ] patients
  [ENTITY         ] postmenopausal
  [ENTITY         ] women
  [ENTITY         ] Adequate
  [ENTITY         

/home/spapadias/.local/lib/python3.12/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


## Cell 8 — NER Summary Table

Compare all three NER backends on the same passage and ground truth.
**What to look for:** Which backend gives the best recall for food/nutrition entity types? That backend should feed the NEL pipeline.

In [9]:
ner_results_all = [
    ("GLiNER biomed-v0.1",        gliner_metrics,         gliner_runtime),
    ("GLiNER large-v2.1 (flat)",  gliner_v2_metrics_flat, gliner_v2_runtime_flat),
    ("GLiNER large-v2.1 (nested)",gliner_v2_metrics_nested, gliner_v2_runtime_nested),
    (f"spaCy ({spacy_model_name or 'N/A'})", spacy_metrics, spacy_runtime),
    ("scispaCy (scibert/lg)",      scispacy_metrics,       scispacy_runtime),
]

print(f"{'Backend':<32} {'Extracted':>9} {'Precision':>10} {'Recall':>8} {'F1':>6} {'Jaccard':>8} {'Runtime':>8}")
print("-" * 88)
for name, m, rt in ner_results_all:
    print(
        f"{name:<32} {m['extracted']:>9} "
        f"{m['precision']:>10.3f} {m['recall']:>8.3f} "
        f"{m['f1']:>6.3f} {m['jaccard']:>8.3f} {rt:>7.2f}s"
    )

Backend                          Extracted  Precision   Recall     F1  Jaccard  Runtime
----------------------------------------------------------------------------------------
GLiNER biomed-v0.1                      39      0.897    0.921  0.909    0.833    7.05s
GLiNER large-v2.1 (flat)                38      0.842    0.842  0.842    0.727    0.42s
GLiNER large-v2.1 (nested)              40      0.825    0.868  0.846    0.733    0.43s
spaCy (en_core_web_trf)                  5      0.800    0.105  0.186    0.103    0.12s
scispaCy (scibert/lg)                   62      0.516    0.842  0.640    0.471    0.03s


In [12]:
# Cell 8b — SciFoodNER NER + NEL via subprocess
# Runs SciFoodNER in the scifoodner conda environment via subprocess.
# No kernel switching needed — results written to JSON and read back here.
# Run time: ~30s for cafeteria + ~30s for foodon on GPU, ~2min on CPU.

import json
import subprocess
from pathlib import Path

SCIFOODNER_PYTHON  = "/mnt/data/makis/conda_envs/scifoodner/bin/python"
SCIFOODNER_SCRIPT  = os.path.join(PROJECT_ROOT, "scripts", "run_scifoodner_inference.py")
SCIFOODNER_OUTPUT  = os.path.join(PROJECT_ROOT, "notebook_artifacts", "scifoodner_results.json")

Path(os.path.dirname(SCIFOODNER_OUTPUT)).mkdir(parents=True, exist_ok=True)

# Write passage and ground truth to a temp input file
input_data = {
    "passage": PASSAGE,
    "ground_truth": list(GROUND_TRUTH),
}
input_path = os.path.join(PROJECT_ROOT, "notebook_artifacts", "scifoodner_input.json")
with open(input_path, "w") as f:
    json.dump(input_data, f)

if not Path(SCIFOODNER_PYTHON).exists():
    print("SKIP: scifoodner environment not found at " + SCIFOODNER_PYTHON)
    print("Install with: conda create --prefix /mnt/data/makis/conda_envs/scifoodner python=3.9")
    scifoodner_available = False
    scifoodner_results   = None
elif not Path(SCIFOODNER_SCRIPT).exists():
    print("SKIP: inference script not found at " + SCIFOODNER_SCRIPT)
    scifoodner_available = False
    scifoodner_results   = None
else:
    print("Running SciFoodNER inference (cafeteria + foodon)...")
    print("This spawns a subprocess in the scifoodner conda environment.")
    t0 = time.time()
    result = subprocess.run(
        [SCIFOODNER_PYTHON, SCIFOODNER_SCRIPT,
         "--input",  input_path,
         "--output", SCIFOODNER_OUTPUT],
        capture_output=True, text=True, timeout=300,
    )
    elapsed = time.time() - t0

    if result.returncode != 0:
        print("SciFoodNER subprocess FAILED (exit code " + str(result.returncode) + ")")
        print("stderr:\n" + result.stderr[-2000:])
        scifoodner_available = False
        scifoodner_results   = None
    else:
        print("SciFoodNER done in " + f"{elapsed:.1f}s")
        if result.stderr:
            print("stderr (info):\n" + result.stderr[-500:])

        with open(SCIFOODNER_OUTPUT) as f:
            scifoodner_results = json.load(f)
        scifoodner_available = True

        F = scifoodner_results["variant_F"]
        G = scifoodner_results["variant_G"]

        extracted_F_loaded = set(F["extracted_entities"])
        extracted_G_loaded = set(G["extracted_entities"])

        col_w = (35, 9, 10, 8, 6, 8, 8)
        header = (
            "Backend".ljust(col_w[0])
            + "Extracted".rjust(col_w[1])
            + "Precision".rjust(col_w[2])
            + "Recall".rjust(col_w[3])
            + "F1".rjust(col_w[4])
            + "Jaccard".rjust(col_w[5])
            + "Runtime".rjust(col_w[6])
        )
        print("\nSciFoodNER NER results:")
        print(header)
        print("-" * 92)

        for name, m, rt in ner_results_all:
            row = (
                name.ljust(col_w[0])
                + str(m["extracted"]).rjust(col_w[1])
                + f"{m['precision']:.3f}".rjust(col_w[2])
                + f"{m['recall']:.3f}".rjust(col_w[3])
                + f"{m['f1']:.3f}".rjust(col_w[4])
                + f"{m['jaccard']:.3f}".rjust(col_w[5])
                + f"{rt:.2f}s".rjust(col_w[6])
            )
            print(row)

        for label, vk in [("SciFoodNER cafeteria (NER)", "variant_F"),
                           ("SciFoodNER foodon (NER part)", "variant_G")]:
            v = scifoodner_results[vk]
            m = v["ner_metrics"]
            row = (
                label.ljust(col_w[0])
                + str(m["extracted"]).rjust(col_w[1])
                + f"{m['precision']:.3f}".rjust(col_w[2])
                + f"{m['recall']:.3f}".rjust(col_w[3])
                + f"{m['f1']:.3f}".rjust(col_w[4])
                + f"{m['jaccard']:.3f}".rjust(col_w[5])
                + f"{v['ner_runtime']:.2f}s".rjust(col_w[6])
            )
            print(row)

Running SciFoodNER inference (cafeteria + foodon)...
This spawns a subprocess in the scifoodner conda environment.
SciFoodNER done in 6.9s
stderr (info):
afeteria inference: 1.56s
Running Variant G (foodon — NER+NEL)...
  Loading foodon model (CUDA=True)...

100%|██████████| 1/1 [00:00<00:00, 232.13it/s]

Running Prediction: 100%|██████████| 1/1 [00:00<00:00, 133.60it/s]
  foodon inference: 0.06s
Results saved to /mnt/data/makis/merlin/LinearRAG/notebook_artifacts/scifoodner_results.json
F: Jaccard=0.089, 11 entities
G: Jaccard=0.048, NEL GT Cov=0.053


SciFoodNER NER results:
Backend                            Extracted Precision  Recall    F1 Jaccard Runtime
--------------------------------------------------------------------------------------------
GLiNER biomed-v0.1                        39     0.897   0.921 0.909   0.833   7.05s
GLiNER large-v2.1 (flat)                  38     0.842   0.842 0.842   0.727   0.42s
GLiNER large-v2.1 (nested)                40     0.825   0.868 0.846

In [13]:
# Verify GLiNER Bio v0.1 warm inference time
# Re-runs the same prediction to isolate cold start from actual inference speed.
# The model is already loaded in memory — this measures pure inference time.

import time

print("GLiNER Bio v0.1 — warm inference timing test")
print("(Model already loaded — no cold start)\n")

# Run 3 times to get stable measurement
for i in range(3):
    t0 = time.time()
    _ = gliner_model.predict_entities(PASSAGE, GLINER_LABELS, threshold=0.4)
    rt = time.time() - t0
    print(f"  Run {i+1}: {rt:.3f}s")

GLiNER Bio v0.1 — warm inference timing test
(Model already loaded — no cold start)

  Run 1: 0.344s
  Run 2: 0.300s
  Run 3: 0.319s


## Cell 9 — NEL Prerequisites Check

Checks which HNSW index files exist and whether CUDA is available.
**What to look for:** Variants requiring missing files or GPU will be skipped in subsequent cells.
Build missing indexes with: `python build_nel_index.py --owl ontology/foodon.owl --encoder {encoder}`

In [14]:
import os

METADATA_PATH = os.path.join(PROJECT_ROOT, "ontology", "foodon_metadata.json")
HNSW_PATHS = {
    "sapbert": os.path.join(PROJECT_ROOT, "ontology", "foodon_hnsw_sapbert.bin"),
    "biolord": os.path.join(PROJECT_ROOT, "ontology", "foodon_hnsw_biolord.bin"),
    "minilm":  os.path.join(PROJECT_ROOT, "ontology", "foodon_hnsw_minilm.bin"),
    "mpnet":   os.path.join(PROJECT_ROOT, "ontology", "foodon_hnsw_mpnet.bin"),
}

metadata_ok = os.path.exists(METADATA_PATH)

print(f"Metadata ({METADATA_PATH}): {'✓ found' if metadata_ok else '✗ missing'}")
print()

status = {}
status["lexical"] = metadata_ok
for enc, path in HNSW_PATHS.items():
    ok = os.path.exists(path) and metadata_ok
    status[enc] = ok
    print(f"Variant B (hnsw/{enc:8s}): {'✓ ready' if ok else f'✗ missing {os.path.basename(path)}'}")

print()
print(f"Variant A (lexical):      {'✓ ready' if status['lexical'] else '✗ missing metadata'}")
print(f"Variant C (hnsw_foodsem): {'✓ ready (sapbert + GPU)' if (status.get('sapbert') and CUDA_AVAILABLE) else '✗ ' + ('no CUDA' if not CUDA_AVAILABLE else 'missing sapbert index')}")
print(f"Variant D (foodsem):      {'✓ ready (GPU)' if CUDA_AVAILABLE else '✗ no CUDA'}")

Metadata (/mnt/data/makis/merlin/LinearRAG/ontology/foodon_metadata.json): ✓ found

Variant B (hnsw/sapbert ): ✓ ready
Variant B (hnsw/biolord ): ✓ ready
Variant B (hnsw/minilm  ): ✓ ready
Variant B (hnsw/mpnet   ): ✓ ready

Variant A (lexical):      ✓ ready
Variant C (hnsw_foodsem): ✗ no CUDA
Variant D (foodsem):      ✗ no CUDA


## Cell 10 — NEL Utility Functions

Helpers for running NEL linkers and printing per-entity results.
- `run_nel_on_entities`: runs a loaded linker over a list of surface forms
- `print_nel_table`: prints per-entity method/confidence/URI table
- `nel_summary_metrics`: computes linked rate and GT coverage

In [15]:
def run_nel_on_entities(linker, entities, context=""):
    results = {}
    t0 = time.time()
    for entity in entities:
        results[entity] = linker.link(entity, context=context)
    runtime = time.time() - t0
    return results, runtime


def print_nel_table(variant_name, results, runtime):
    print(f"\n{'='*80}")
    print(f"NEL: {variant_name}  ({runtime:.2f}s total, {runtime/max(len(results),1)*1000:.1f}ms/entity)")
    print(f"{'='*80}")
    print(f"{'Entity':<30} {'Method':>12} {'Conf':>6}  {'URI segment':<25} {'Canonical label'}")
    print("-" * 100)
    for entity, res in sorted(results.items()):
        uri_seg = res.uri.split("/")[-1] if res.uri else "NIL"
        canonical = res.canonical_label or ""
        print(
            f"{entity:<30} {res.method:>12} {res.confidence:>6.3f}  "
            f"{uri_seg:<25} {canonical}"
        )


def nel_summary_metrics(results, ground_truth):
    linked = {e for e, r in results.items() if r.uri is not None}
    nil = {e for e, r in results.items() if r.uri is None}
    gt_lower = {g.lower() for g in ground_truth}
    gt_covered = {e for e in linked if e.lower() in gt_lower}
    linked_rate = len(linked) / len(results) if results else 0.0
    gt_coverage = len(gt_covered) / len(ground_truth) if ground_truth else 0.0
    return {
        "linked": len(linked),
        "nil": len(nil),
        "linked_rate": linked_rate,
        "gt_coverage": gt_coverage,
        "gt_covered": gt_covered,
    }


print("NEL utility functions defined.")

NEL utility functions defined.


## Cell 11 — Variant A: Lexical NEL

RapidFuzz token_sort_ratio, threshold=90. No model. Runs in milliseconds.
**What to look for:** High precision on exact/near-exact matches (olive oil, salmon, iron).
Expected NIL for abbreviations and paraphrases.

In [16]:
from src.nel import LexicalNELLinker

if not status["lexical"]:
    print("SKIP: metadata not found. Run build_nel_index.py first.")
else:
    lexical_linker = LexicalNELLinker(
        metadata_path=METADATA_PATH,
        threshold=90.0,
    )
    t_load = time.time()
    lexical_linker.load()
    print(f"Load time: {time.time() - t_load:.2f}s")

    print("\n--- Variant A on Ground Truth entities ---")
    gt_results_A, gt_runtime_A = run_nel_on_entities(lexical_linker, sorted(GROUND_TRUTH), context=PASSAGE)
    print_nel_table("Variant A (Lexical) — GT entities", gt_results_A, gt_runtime_A)
    gt_metrics_A = nel_summary_metrics(gt_results_A, GROUND_TRUTH)
    print(f"\nSummary: linked={gt_metrics_A['linked']}, NIL={gt_metrics_A['nil']}, "
          f"linked_rate={gt_metrics_A['linked_rate']:.3f}, "
          f"GT_coverage={gt_metrics_A['gt_coverage']:.3f}")

    print("\n--- Variant A on GLiNER-extracted entities ---")
    gliner_results_A, gliner_runtime_A = run_nel_on_entities(lexical_linker, sorted(gliner_extracted), context=PASSAGE)
    print_nel_table("Variant A (Lexical) — GLiNER entities", gliner_results_A, gliner_runtime_A)
    gliner_metrics_A = nel_summary_metrics(gliner_results_A, GROUND_TRUTH)
    print(f"\nSummary: linked={gliner_metrics_A['linked']}, NIL={gliner_metrics_A['nil']}, "
          f"GT_coverage={gliner_metrics_A['gt_coverage']:.3f}")

Load time: 0.07s

--- Variant A on Ground Truth entities ---

NEL: Variant A (Lexical) — GT entities  (0.44s total, 11.7ms/entity)
Entity                               Method   Conf  URI segment               Canonical label
----------------------------------------------------------------------------------------------------
BMI                                     nil  0.600  NIL                       
DASH diet                           lexical  1.000  ONS_1000037               DASH diet
Greece                              lexical  1.000  GAZ_00002945              Greece
Greek yogurt                        lexical  1.000  FOODON_00004409           greek yogurt
HbA1c                                   nil  0.615  NIL                       
Italy                               lexical  1.000  GAZ_00002650              Italy
Japan                               lexical  1.000  GAZ_00002747              Japan
LDL cholesterol                         nil  0.846  NIL                       
Medit

## Cell 12 — Variant B: HNSW + SapBERT

SapBERT is fine-tuned on UMLS biomedical synonyms — best encoder for abbreviation/scientific name recall.
Decision: score gap heuristic (min_sim=0.70, gap=0.08).
**What to look for:** Does SapBERT catch entities missed by lexical (HbA1c, polyphenols, carotenoids)?

In [17]:
from src.nel import HNSWNELLinker

if not status["sapbert"]:
    print("SKIP: foodon_hnsw_sapbert.bin not found.")
    print("Build with: python build_nel_index.py --owl ontology/foodon.owl --encoder sapbert")
    gt_metrics_B_sap = None
else:
    linker_B_sap = HNSWNELLinker(
        index_path=HNSW_PATHS["sapbert"],
        metadata_path=METADATA_PATH,
        encoder="sapbert",
        top_k=1,
        min_sim=0.70,
    )
    t_load = time.time()
    linker_B_sap.load()
    print(f"Load time: {time.time() - t_load:.2f}s")

    gt_results_B_sap, gt_runtime_B_sap = run_nel_on_entities(linker_B_sap, sorted(GROUND_TRUTH), context=PASSAGE)
    print_nel_table("Variant B (SapBERT) — GT entities", gt_results_B_sap, gt_runtime_B_sap)
    gt_metrics_B_sap = nel_summary_metrics(gt_results_B_sap, GROUND_TRUTH)
    print(f"\nSummary: linked={gt_metrics_B_sap['linked']}, NIL={gt_metrics_B_sap['nil']}, "
          f"linked_rate={gt_metrics_B_sap['linked_rate']:.3f}, "
          f"GT_coverage={gt_metrics_B_sap['gt_coverage']:.3f}")

    gliner_results_B_sap, gliner_runtime_B_sap = run_nel_on_entities(linker_B_sap, sorted(gliner_extracted), context=PASSAGE)
    gliner_metrics_B_sap = nel_summary_metrics(gliner_results_B_sap, GROUND_TRUTH)

Load time: 1.79s

NEL: Variant B (SapBERT) — GT entities  (0.75s total, 19.6ms/entity)
Entity                               Method   Conf  URI segment               Canonical label
----------------------------------------------------------------------------------------------------
BMI                                     nil  0.605  NIL                       
DASH diet                              hnsw  1.000  ONS_1000037               DASH diet
Greece                                 hnsw  1.000  GAZ_00002945              Greece
Greek yogurt                           hnsw  1.000  FOODON_00004409           greek yogurt
HbA1c                                   nil  0.625  NIL                       
Italy                                  hnsw  1.000  GAZ_00002650              Italy
Japan                                  hnsw  1.000  GAZ_00002747              Japan
LDL cholesterol                        hnsw  0.799  CHEBI_39026               low-density lipoprotein
Mediterranean diet        

In [18]:
# Cell 12b — Variant B': HNSW (SapBERT) + MiniLM cross-encoder reranker
# Two-stage pipeline: SapBERT HNSW retrieves top-10, cross-encoder reranks.
# Uses cross-encoder/ms-marco-MiniLM-L-6-v2 — trained on MS-MARCO/NLI,
# complementary to SapBERT (UMLS synonyms). Different models per stage is
# essential: same model retrieval+reranking = no improvement.

from src.nel import HNSWCrossEncoderNELLinker

if not status["sapbert"]:
    print("SKIP: foodon_hnsw_sapbert.bin not found.")
    gt_metrics_B_prime = None
else:
    linker_B_prime = HNSWCrossEncoderNELLinker(
        index_path=HNSW_PATHS["sapbert"],
        metadata_path=METADATA_PATH,
        encoder="sapbert",
        top_k=10,
        rerank_threshold=0.0,
        cross_encoder_model="cross-encoder/ms-marco-MiniLM-L-12-v2",
    )
    t_load = time.time()
    linker_B_prime.load()
    print(f"Load time: {time.time() - t_load:.2f}s")

    gt_results_B_prime, gt_runtime_B_prime = run_nel_on_entities(
        linker_B_prime, sorted(GROUND_TRUTH), context=PASSAGE
    )
    print_nel_table("Variant B' (SapBERT+CrossEncoder) — GT entities",
                    gt_results_B_prime, gt_runtime_B_prime)
    gt_metrics_B_prime = nel_summary_metrics(gt_results_B_prime, GROUND_TRUTH)
    print(f"\nSummary: linked={gt_metrics_B_prime['linked']}, "
          f"NIL={gt_metrics_B_prime['nil']}, "
          f"linked_rate={gt_metrics_B_prime['linked_rate']:.3f}, "
          f"GT_coverage={gt_metrics_B_prime['gt_coverage']:.3f}")

    gliner_results_B_prime, gliner_runtime_B_prime = run_nel_on_entities(
        linker_B_prime, sorted(gliner_extracted), context=PASSAGE
    )
    gliner_metrics_B_prime = nel_summary_metrics(gliner_results_B_prime, GROUND_TRUTH)


Load time: 21.97s

NEL: Variant B' (SapBERT+CrossEncoder) — GT entities  (1.47s total, 38.6ms/entity)
Entity                               Method   Conf  URI segment               Canonical label
----------------------------------------------------------------------------------------------------
BMI                                     nil -10.979  NIL                       
DASH diet                      hnsw_crossencoder  6.324  ONS_2000037               obsolete: DASH diet dietary pattern
Greece                         hnsw_crossencoder  6.436  GAZ_00002945              Greece
Greek yogurt                   hnsw_crossencoder  7.784  FOODON_00004409           greek yogurt
HbA1c                                   nil -9.975  NIL                       
Italy                          hnsw_crossencoder  8.128  HANCESTRO_0307            Italian
Japan                          hnsw_crossencoder  4.683  HANCESTRO_0754            Japanese in Tokyo, Japan (1KGP)
LDL cholesterol                  

## Cell 13 — Variant B: HNSW + BioLORD

BioLORD is trained on SNOMED/MeSH concept descriptions — strong on clinical/biomedical terms.
**What to look for:** Compare vs SapBERT on nutrition-specific terms. BioLORD may rank differently for food entities.

In [19]:
if not status["biolord"]:
    print("SKIP: foodon_hnsw_biolord.bin not found.")
    print("Build with: python build_nel_index.py --owl ontology/foodon.owl --encoder biolord")
    gt_metrics_B_bio = None
else:
    linker_B_bio = HNSWNELLinker(
        index_path=HNSW_PATHS["biolord"],
        metadata_path=METADATA_PATH,
        encoder="biolord",
        top_k=1, min_sim=0.70,
    )
    t_load = time.time()
    linker_B_bio.load()
    print(f"Load time: {time.time() - t_load:.2f}s")

    gt_results_B_bio, gt_runtime_B_bio = run_nel_on_entities(linker_B_bio, sorted(GROUND_TRUTH), context=PASSAGE)
    print_nel_table("Variant B (BioLORD) — GT entities", gt_results_B_bio, gt_runtime_B_bio)
    gt_metrics_B_bio = nel_summary_metrics(gt_results_B_bio, GROUND_TRUTH)
    print(f"\nSummary: linked={gt_metrics_B_bio['linked']}, NIL={gt_metrics_B_bio['nil']}, "
          f"GT_coverage={gt_metrics_B_bio['gt_coverage']:.3f}")

    gliner_results_B_bio, gliner_runtime_B_bio = run_nel_on_entities(linker_B_bio, sorted(gliner_extracted), context=PASSAGE)
    gliner_metrics_B_bio = nel_summary_metrics(gliner_results_B_bio, GROUND_TRUTH)

Load time: 2.60s

NEL: Variant B (BioLORD) — GT entities  (0.58s total, 15.3ms/entity)
Entity                               Method   Conf  URI segment               Canonical label
----------------------------------------------------------------------------------------------------
BMI                                     nil  0.570  NIL                       
DASH diet                              hnsw  1.000  ONS_1000037               DASH diet
Greece                                 hnsw  1.000  GAZ_00002945              Greece
Greek yogurt                           hnsw  1.000  FOODON_00004409           greek yogurt
HbA1c                                  hnsw  0.774  CHEBI_35143               hemoglobin
Italy                                  hnsw  1.000  GAZ_00002650              Italy
Japan                                  hnsw  1.000  GAZ_00002747              Japan
LDL cholesterol                        hnsw  0.943  CHEBI_39026               low-density lipoprotein
Mediterranean di

In [22]:
# Cell 13b — Variant B': HNSW (BioLORD) + cross-encoder reranker
from src.nel import HNSWCrossEncoderNELLinker

if not status["biolord"]:
    print("SKIP: foodon_hnsw_biolord.bin not found.")
    gt_metrics_B_prime_bio = None
else:
    linker_B_prime_bio = HNSWCrossEncoderNELLinker(
        index_path=HNSW_PATHS["biolord"],
        metadata_path=METADATA_PATH,
        encoder="biolord",
        top_k=10,
        rerank_threshold=0.0,
        cross_encoder_model="cross-encoder/ms-marco-MiniLM-L-12-v2",
    )
    t_load = time.time()
    linker_B_prime_bio.load()
    print("Load time: " + f"{time.time() - t_load:.2f}s")

    gt_results_B_prime_bio, gt_runtime_B_prime_bio = run_nel_on_entities(
        linker_B_prime_bio, sorted(GROUND_TRUTH), context=PASSAGE
    )
    print_nel_table("Variant B' (BioLORD+CrossEncoder) — GT entities",
                    gt_results_B_prime_bio, gt_runtime_B_prime_bio)
    gt_metrics_B_prime_bio = nel_summary_metrics(gt_results_B_prime_bio, GROUND_TRUTH)
    linked  = gt_metrics_B_prime_bio["linked"]
    nil     = gt_metrics_B_prime_bio["nil"]
    gt_cov  = gt_metrics_B_prime_bio["gt_coverage"]
    print("\nSummary: linked=" + str(linked) + ", NIL=" + str(nil) + ", GT_coverage=" + f"{gt_cov:.3f}")

    gliner_results_B_prime_bio, gliner_runtime_B_prime_bio = run_nel_on_entities(
        linker_B_prime_bio, sorted(gliner_extracted), context=PASSAGE
    )
    gliner_metrics_B_prime_bio = nel_summary_metrics(gliner_results_B_prime_bio, GROUND_TRUTH)

Load time: 8.06s

NEL: Variant B' (BioLORD+CrossEncoder) — GT entities  (1.45s total, 38.1ms/entity)
Entity                               Method   Conf  URI segment               Canonical label
----------------------------------------------------------------------------------------------------
BMI                                     nil -10.672  NIL                       
DASH diet                      hnsw_crossencoder  6.324  ONS_2000037               obsolete: DASH diet dietary pattern
Greece                         hnsw_crossencoder  6.436  GAZ_00002945              Greece
Greek yogurt                   hnsw_crossencoder  7.784  FOODON_00004409           greek yogurt
HbA1c                                   nil -9.391  NIL                       
Italy                          hnsw_crossencoder  8.128  HANCESTRO_0307            Italian
Japan                          hnsw_crossencoder  5.203  HANCESTRO_0717            Japanese in Japan (HGDP)
LDL cholesterol                         n

## Cell 14 — Variant B: HNSW + MiniLM

MiniLM is a general-purpose encoder — fast, small, not domain-adapted.
**What to look for:** General-purpose baseline. Expected lower recall on biomedical abbreviations.

In [24]:
if not status["minilm"]:
    print("SKIP: foodon_hnsw_minilm.bin not found.")
    print("Build with: python build_nel_index.py --owl ontology/foodon.owl --encoder minilm")
    gt_metrics_B_mini = None
else:
    linker_B_mini = HNSWNELLinker(
        index_path=HNSW_PATHS["minilm"],
        metadata_path=METADATA_PATH,
        encoder="minilm",
        top_k=1, min_sim=0.70,
    )
    t_load = time.time()
    linker_B_mini.load()
    print(f"Load time: {time.time() - t_load:.2f}s")

    gt_results_B_mini, gt_runtime_B_mini = run_nel_on_entities(linker_B_mini, sorted(GROUND_TRUTH), context=PASSAGE)
    print_nel_table("Variant B (MiniLM) — GT entities", gt_results_B_mini, gt_runtime_B_mini)
    gt_metrics_B_mini = nel_summary_metrics(gt_results_B_mini, GROUND_TRUTH)
    print(f"\nSummary: linked={gt_metrics_B_mini['linked']}, NIL={gt_metrics_B_mini['nil']}, "
          f"GT_coverage={gt_metrics_B_mini['gt_coverage']:.3f}")

    gliner_results_B_mini, gliner_runtime_B_mini = run_nel_on_entities(linker_B_mini, sorted(gliner_extracted), context=PASSAGE)
    gliner_metrics_B_mini = nel_summary_metrics(gliner_results_B_mini, GROUND_TRUTH)

Load time: 2.56s

NEL: Variant B (MiniLM) — GT entities  (0.20s total, 5.1ms/entity)
Entity                               Method   Conf  URI segment               Canonical label
----------------------------------------------------------------------------------------------------
BMI                                     nil  0.512  NIL                       
DASH diet                              hnsw  1.000  ONS_1000037               DASH diet
Greece                                 hnsw  1.000  GAZ_00002945              Greece
Greek yogurt                           hnsw  1.000  FOODON_00004409           greek yogurt
HbA1c                                   nil  0.535  NIL                       
Italy                                  hnsw  1.000  GAZ_00002650              Italy
Japan                                  hnsw  1.000  GAZ_00002747              Japan
LDL cholesterol                        hnsw  0.716  CHEBI_16113               cholesterol
Mediterranean diet                     h

In [29]:
# Cell 14b — Variant B': HNSW (MiniLM bi-encoder) + cross-encoder reranker
# Retrieval: all-MiniLM-L6-v2 (bi-encoder, general purpose)
# Reranking: ms-marco-MiniLM-L-12-v2 (cross-encoder) — different model.
from src.nel import HNSWCrossEncoderNELLinker

if not status["minilm"]:
    print("SKIP: foodon_hnsw_minilm.bin not found.")
    gt_metrics_B_prime_mini = None
else:
    linker_B_prime_mini = HNSWCrossEncoderNELLinker(
        index_path=HNSW_PATHS["minilm"],
        metadata_path=METADATA_PATH,
        encoder="minilm",
        top_k=10,
        rerank_threshold=0.0,
        cross_encoder_model="cross-encoder/ms-marco-MiniLM-L-12-v2",
    )
    t_load = time.time()
    linker_B_prime_mini.load()
    print("Load time: " + f"{time.time() - t_load:.2f}s")

    gt_results_B_prime_mini, gt_runtime_B_prime_mini = run_nel_on_entities(
        linker_B_prime_mini, sorted(GROUND_TRUTH), context=PASSAGE
    )
    print_nel_table("Variant B' (MiniLM+CrossEncoder) — GT entities",
                    gt_results_B_prime_mini, gt_runtime_B_prime_mini)
    gt_metrics_B_prime_mini = nel_summary_metrics(gt_results_B_prime_mini, GROUND_TRUTH)
    linked = gt_metrics_B_prime_mini["linked"]
    nil    = gt_metrics_B_prime_mini["nil"]
    gt_cov = gt_metrics_B_prime_mini["gt_coverage"]
    print("\nSummary: linked=" + str(linked) + ", NIL=" + str(nil) + ", GT_coverage=" + f"{gt_cov:.3f}")

    gliner_results_B_prime_mini, gliner_runtime_B_prime_mini = run_nel_on_entities(
        linker_B_prime_mini, sorted(gliner_extracted), context=PASSAGE
    )
    gliner_metrics_B_prime_mini = nel_summary_metrics(gliner_results_B_prime_mini, GROUND_TRUTH)

Load time: 5.65s

NEL: Variant B' (MiniLM+CrossEncoder) — GT entities  (1.02s total, 26.9ms/entity)
Entity                               Method   Conf  URI segment               Canonical label
----------------------------------------------------------------------------------------------------
BMI                                     nil -10.534  NIL                       
DASH diet                      hnsw_crossencoder  6.324  ONS_2000037               obsolete: DASH diet dietary pattern
Greece                         hnsw_crossencoder  6.859  NCIT_C25464               Country
Greek yogurt                   hnsw_crossencoder  7.784  FOODON_00004409           greek yogurt
HbA1c                                   nil -5.212  NIL                       
Italy                          hnsw_crossencoder  8.128  HANCESTRO_0307            Italian
Japan                          hnsw_crossencoder  6.293  GAZ_00000465              Asia
LDL cholesterol                         nil -4.721  NIL      

## Cell 15 — Variant B: HNSW + MPNET

MPNET is the same encoder LinearRAG uses for passage/entity embeddings.
**What to look for:** Does consistency with LinearRAG's embedding space help or hurt NEL quality?

In [30]:
if not status["mpnet"]:
    print("SKIP: foodon_hnsw_mpnet.bin not found.")
    print("Build with: python build_nel_index.py --owl ontology/foodon.owl --encoder mpnet")
    gt_metrics_B_mpnet = None
else:
    linker_B_mpnet = HNSWNELLinker(
        index_path=HNSW_PATHS["mpnet"],
        metadata_path=METADATA_PATH,
        encoder="mpnet",
        top_k=1, min_sim=0.70,
    )
    t_load = time.time()
    linker_B_mpnet.load()
    print(f"Load time: {time.time() - t_load:.2f}s")

    gt_results_B_mpnet, gt_runtime_B_mpnet = run_nel_on_entities(linker_B_mpnet, sorted(GROUND_TRUTH), context=PASSAGE)
    print_nel_table("Variant B (MPNET) — GT entities", gt_results_B_mpnet, gt_runtime_B_mpnet)
    gt_metrics_B_mpnet = nel_summary_metrics(gt_results_B_mpnet, GROUND_TRUTH)
    print(f"\nSummary: linked={gt_metrics_B_mpnet['linked']}, NIL={gt_metrics_B_mpnet['nil']}, "
          f"GT_coverage={gt_metrics_B_mpnet['gt_coverage']:.3f}")

    gliner_results_B_mpnet, gliner_runtime_B_mpnet = run_nel_on_entities(linker_B_mpnet, sorted(gliner_extracted), context=PASSAGE)
    gliner_metrics_B_mpnet = nel_summary_metrics(gliner_results_B_mpnet, GROUND_TRUTH)

Load time: 2.63s

NEL: Variant B (MPNET) — GT entities  (0.59s total, 15.6ms/entity)
Entity                               Method   Conf  URI segment               Canonical label
----------------------------------------------------------------------------------------------------
BMI                                     nil  0.584  NIL                       
DASH diet                              hnsw  1.000  ONS_1000037               DASH diet
Greece                                 hnsw  1.000  GAZ_00002945              Greece
Greek yogurt                           hnsw  1.000  FOODON_00004409           greek yogurt
HbA1c                                   nil  0.492  NIL                       
Italy                                  hnsw  1.000  GAZ_00002650              Italy
Japan                                  hnsw  1.000  GAZ_00002747              Japan
LDL cholesterol                        hnsw  0.805  CHEBI_16113               cholesterol
Mediterranean diet                     h

In [ ]:
# Cell 15b — Variant B': HNSW (MPNET) + cross-encoder reranker
from src.nel import HNSWCrossEncoderNELLinker

if not status["mpnet"]:
    print("SKIP: foodon_hnsw_mpnet.bin not found.")
    gt_metrics_B_prime_mpnet = None
else:
    linker_B_prime_mpnet = HNSWCrossEncoderNELLinker(
        index_path=HNSW_PATHS["mpnet"],
        metadata_path=METADATA_PATH,
        encoder="mpnet",
        top_k=10,
        rerank_threshold=0.0,
        cross_encoder_model="cross-encoder/ms-marco-MiniLM-L-12-v2",
    )
    t_load = time.time()
    linker_B_prime_mpnet.load()
    print("Load time: " + f"{time.time() - t_load:.2f}s")

    gt_results_B_prime_mpnet, gt_runtime_B_prime_mpnet = run_nel_on_entities(
        linker_B_prime_mpnet, sorted(GROUND_TRUTH), context=PASSAGE
    )
    print_nel_table("Variant B' (MPNET+CrossEncoder) — GT entities",
                    gt_results_B_prime_mpnet, gt_runtime_B_prime_mpnet)
    gt_metrics_B_prime_mpnet = nel_summary_metrics(gt_results_B_prime_mpnet, GROUND_TRUTH)
    linked = gt_metrics_B_prime_mpnet["linked"]
    nil    = gt_metrics_B_prime_mpnet["nil"]
    gt_cov = gt_metrics_B_prime_mpnet["gt_coverage"]
    print("\nSummary: linked=" + str(linked) + ", NIL=" + str(nil) + ", GT_coverage=" + f"{gt_cov:.3f}")

    gliner_results_B_prime_mpnet, gliner_runtime_B_prime_mpnet = run_nel_on_entities(
        linker_B_prime_mpnet, sorted(gliner_extracted), context=PASSAGE
    )
    gliner_metrics_B_prime_mpnet = nel_summary_metrics(gliner_results_B_prime_mpnet, GROUND_TRUTH)

Load time: 5.73s

NEL: Variant B' (MPNET+CrossEncoder) — GT entities  (1.47s total, 38.6ms/entity)
Entity                               Method   Conf  URI segment               Canonical label
----------------------------------------------------------------------------------------------------
BMI                                     nil -10.122  NIL                       
DASH diet                      hnsw_crossencoder  6.324  ONS_2000037               obsolete: DASH diet dietary pattern
Greece                         hnsw_crossencoder  6.436  GAZ_00002945              Greece
Greek yogurt                   hnsw_crossencoder  7.784  FOODON_00004409           greek yogurt
HbA1c                                   nil -9.363  NIL                       
Italy                          hnsw_crossencoder  8.128  HANCESTRO_0307            Italian
Japan                          hnsw_crossencoder  6.293  GAZ_00000465              Asia
LDL cholesterol                         nil -4.721  NIL        

In [32]:
# Run this as a quick cell before Cell 16
import torch
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free = torch.cuda.mem_get_info(i)[0] / 1024**3
        total = torch.cuda.mem_get_info(i)[1] / 1024**3
        print(f"GPU {i}: {free:.1f}GB free / {total:.1f}GB total")

In [33]:
# Release GPU memory
import gc
try:
    del linker_D
except NameError:
    pass
gc.collect()
try:
    torch.cuda.empty_cache()
    free = torch.cuda.mem_get_info(0)[0] / 1024**3
    print(f"GPU 0 free after cleanup: {free:.1f}GB")
except RuntimeError as e:
    print(f"GPU memory info unavailable: {e}")
    print("Continuing — model deleted from Python memory.")

GPU memory info unavailable: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.
Continuing — model deleted from Python memory.


## Cell 16 — Variant C: HNSW + FoodSEM (GPU required)

SapBERT HNSW retrieves top-10 candidates. FoodSEM Llama-3-8B (4-bit) reranks them.
**What to look for:** Does FoodSEM improve precision over the gap heuristic? Watch NIL decisions — FoodSEM is conservative.
Runtime will be ~1-3s per entity on A100.

In [34]:
# Cell 16 — Variant C: HNSW (SapBERT) + FoodSEM reranker
# Overrides HF_HOME to shared cache where Llama is stored, then enables
# offline mode so no network calls are made for the gated model.
# This override affects only this cell and Cell 17 — all other cells
# use the personal cache set in Cell 2.

# Verify link_batch() delegation is active
import inspect
from src.nel import HNSWFoodSEMNELLinker, FoodSEMNELLinker

c_link_src = inspect.getsource(HNSWFoodSEMNELLinker.link)
d_link_src = inspect.getsource(FoodSEMNELLinker.link)

# Variant C: link() should use per-entity _foodsem_rerank(), NOT delegate to link_batch()
if "_foodsem_rerank" in c_link_src:
    print("✓ HNSWFoodSEMNELLinker.link() uses per-entity _foodsem_rerank() — correct")
else:
    print("✗ HNSWFoodSEMNELLinker.link() NOT using _foodsem_rerank() — fix src/nel.py")

# Variant D: link() should delegate to link_batch()
if "link_batch" in d_link_src:
    print("✓ FoodSEMNELLinker.link() delegates to link_batch() — correct")
else:
    print("✗ FoodSEMNELLinker.link() NOT delegating — fix src/nel.py")

LLAMA_LOCAL   = "/mnt/data/makis/hf_cache/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2"
FOODSEM_LOCAL = "/mnt/data/makis/hf_cache/models--Matej--FoodSEM-LLM/snapshots/33e76a1226fdc604e5e539fdb38676593a71b2f5"

import os
import torch
from pathlib import Path

# Switch to shared cache for Llama + enable offline mode
os.environ["HF_HOME"]              = "/mnt/data/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"]   = "/mnt/data/huggingface_cache"
# os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"]  = "1"
print(f"Cache: {os.environ['HF_HOME']} (offline mode ON)")

INDEX_PATH    = os.path.join(PROJECT_ROOT, "ontology", "foodon_hnsw_sapbert.bin")
METADATA_PATH = os.path.join(PROJECT_ROOT, "ontology", "foodon_metadata.json")

if not CUDA_AVAILABLE:
    print("SKIP: Variant C requires CUDA GPU.")
    gt_metrics_C, gliner_metrics_C = None, None
    gt_results_C, gliner_results_C = {}, {}
elif not Path(INDEX_PATH).exists():
    print(f"SKIP: SapBERT index not found at {INDEX_PATH}")
    gt_metrics_C, gliner_metrics_C = None, None
    gt_results_C, gliner_results_C = {}, {}
else:
    from src.nel import HNSWFoodSEMNELLinker

    print("\nLoading Variant C (HNSW + FoodSEM)...")
    print("Loading 8B LLM in 4-bit — takes a few minutes on first load.")
    t0 = time.time()
    linker_C = HNSWFoodSEMNELLinker(
        index_path=INDEX_PATH,
        metadata_path=METADATA_PATH,
        top_k=10,
        local_base_model_path=LLAMA_LOCAL,
        local_adapter_path=FOODSEM_LOCAL,
    )
    linker_C.load()
    print(f"Loaded in {time.time() - t0:.2f}s\n")

    ###################
    # Diagnostic: see raw FoodSEM output for a small test batch
    print("Running diagnostic batch call with 3 entities...")
    test_entities = ["olive oil", "vitamin D", "salmon"]

    # Manually replicate what link_batch does to see the raw output
    candidates_test = {}
    for sf in test_entities:
        candidates_test[sf] = linker_C._retrieve_candidates(sf)
        print(f"  {sf} top-2 candidates:")
        for c in candidates_test[sf][:2]:
            print(f"    {c['canonical_label']} ({c['uri'].split('/')[-1]})")

    # Build the prompt
    candidate_block = ""
    for sf in test_entities:
        cands = candidates_test[sf]
        candidate_str = "; ".join(
            f"{c['canonical_label']} ({c['uri'].split('/')[-1]})"
            for c in cands[:2]
        )
        candidate_block += f"\n{sf}: {candidate_str}"

    user_prompt = (
        f"Might I trouble you to connect the extracted food entities to a "
        f"FoodOn ontology, if possible?\n"
        f"For each entity, pick the best matching URI from its candidates, "
        f"or reply NIL if none match.\n"
        f"Entities and candidates:{candidate_block}"
    )

    print(f"\nPrompt sent to FoodSEM:\n{user_prompt}")
    print(f"\nPrompt token count (approx): {len(user_prompt.split())}")

    # Run FoodSEM
    messages = [{"role": "user", "content": user_prompt}]
    prompt = linker_C._foodsem_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = linker_C._foodsem_tokenizer(
        [prompt], return_tensors="pt", padding=True, 
        truncation=True, max_length=768
    ).to(linker_C._foodsem_model.device)

    import torch
    with torch.no_grad():
        generated_ids = linker_C._foodsem_model.generate(
            **inputs, max_new_tokens=512, do_sample=False
        )
    answer = linker_C._foodsem_tokenizer.batch_decode(
        generated_ids[:, inputs["input_ids"].shape[1]:]
    )[0].split("<|eot_id|>")[0].strip()

    print(f"\nFoodSEM raw answer:\n{answer}")
    ###################

    # Single batch call — all GT entities in one FoodSEM prompt
    print("--- Variant C on Ground Truth entities (single batch call) ---")
    t0 = time.time()
    gt_results_C = linker_C.link_batch(sorted(GROUND_TRUTH), context=PASSAGE)
    gt_runtime_C = time.time() - t0
    print_nel_table("Variant C (HNSW+FoodSEM) — GT batch", gt_results_C, gt_runtime_C)
    gt_metrics_C = nel_summary_metrics(gt_results_C, GROUND_TRUTH)
    print(
        f"\nSummary: linked={gt_metrics_C['linked']}, NIL={gt_metrics_C['nil']}, "
        f"linked_rate={gt_metrics_C['linked_rate']:.3f}, "
        f"GT_coverage={gt_metrics_C['gt_coverage']:.3f}"
    )

    # Single batch call — all GLiNER entities in one FoodSEM prompt
    print("\n--- Variant C on GLiNER-extracted entities (single batch call) ---")
    t0 = time.time()
    gliner_results_C = linker_C.link_batch(sorted(gliner_extracted), context=PASSAGE)
    gliner_runtime_C = time.time() - t0
    gliner_metrics_C = nel_summary_metrics(gliner_results_C, GROUND_TRUTH)
    print(
        f"Summary: linked={gliner_metrics_C['linked']}, NIL={gliner_metrics_C['nil']}, "
        f"GT_coverage={gliner_metrics_C['gt_coverage']:.3f}, "
        f"runtime={gliner_runtime_C:.2f}s"
    )


✓ HNSWFoodSEMNELLinker.link() uses per-entity _foodsem_rerank() — correct
✓ FoodSEMNELLinker.link() delegates to link_batch() — correct
Cache: /mnt/data/huggingface_cache (offline mode ON)
SKIP: Variant C requires CUDA GPU.


## Cell 17 — Variant D refined: FoodSEM + label-aware ordering (Idea 1 only)

Ablation: Idea 1 only (label-aware batch ordering). Original URI parser — no suffix extraction, no metadata validation.
Baseline for isolating the contribution of robust URI parsing (Idea 2).


In [ ]:
# Cell 17 — Variant D: FoodSEM alone
# Shared cache + offline mode already set by Cell 16.
# Re-asserts here in case Cell 17 is run independently.
# WARNING: URIs NOT validated against candidate list — hallucination possible.
# Research comparison only. 

LLAMA_LOCAL   = "/mnt/data/makis/hf_cache/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2"
FOODSEM_LOCAL = "/mnt/data/makis/hf_cache/models--Matej--FoodSEM-LLM/snapshots/33e76a1226fdc604e5e539fdb38676593a71b2f5"

import os
import torch

os.environ["HF_HOME"]              = "/mnt/data/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"]   = "/mnt/data/huggingface_cache"
# os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"]  = "1"
print(f"Cache: {os.environ['HF_HOME']} (offline mode ON)")
print("WARNING: Variant D URIs are NOT validated — hallucination possible.\n")

if not CUDA_AVAILABLE:
    print("SKIP: Variant D requires CUDA GPU.")
    gt_metrics_D, gliner_metrics_D = None, None
    gt_results_D, gliner_results_D = {}, {}
else:
    from src.nel import FoodSEMNELLinker

    print("Loading Variant D (FoodSEM alone)...")
    t0 = time.time()
    METADATA_PATH = os.path.join(PROJECT_ROOT, "ontology", "foodon_metadata.json")

    linker_D = FoodSEMNELLinker(
        local_base_model_path=LLAMA_LOCAL,
        local_adapter_path=FOODSEM_LOCAL,
        batch_size=8,
        metadata_path=METADATA_PATH,
        use_robust_uri_parsing=False,  # ablation: Idea 1 only (label ordering, original URI parser)
    )
    linker_D.load()

    # Pass GLiNER labels to enable label-aware batch ordering
    # gliner_entities_raw contains (text, label, score) tuples from Cell 5a
    linker_D.entity_labels = {
        ent["text"]: ent["label"]
        for ent in gliner_entities_raw
    }
    print(f"Entity labels set: {len(linker_D.entity_labels)} entities")
    print("Label-aware ordering: food/nutrient entities will be processed first in each batch")
    print(f"Loaded in {time.time() - t0:.2f}s\n")

    # Single batch call — all GT entities in one FoodSEM prompt
    print("--- Variant D on Ground Truth entities (single batch call) ---")
    t0 = time.time()
    gt_results_D = linker_D.link_batch(sorted(GROUND_TRUTH), context=PASSAGE)
    gt_runtime_D = time.time() - t0
    print_nel_table("Variant D (FoodSEM alone) — GT batch", gt_results_D, gt_runtime_D)
    gt_metrics_D = nel_summary_metrics(gt_results_D, GROUND_TRUTH)
    print(
        f"\nSummary: linked={gt_metrics_D['linked']}, NIL={gt_metrics_D['nil']}, "
        f"linked_rate={gt_metrics_D['linked_rate']:.3f}, "
        f"GT_coverage={gt_metrics_D['gt_coverage']:.3f}"
    )
    print("NOTE: confidence=1.0 is artificial — FoodSEM generative mode has no calibrated scores.")

    # Single batch call — all GLiNER entities in one FoodSEM prompt
    print("\n--- Variant D on GLiNER-extracted entities (single batch call) ---")
    t0 = time.time()
    gliner_results_D = linker_D.link_batch(sorted(gliner_extracted), context=PASSAGE)
    gliner_runtime_D = time.time() - t0
    gliner_metrics_D = nel_summary_metrics(gliner_results_D, GROUND_TRUTH)
    print(
        f"Summary: linked={gliner_metrics_D['linked']}, NIL={gliner_metrics_D['nil']}, "
        f"GT_coverage={gliner_metrics_D['gt_coverage']:.3f}, "
        f"runtime={gliner_runtime_D:.2f}s"
    )

# Release GPU memory
import gc
try:
    del linker_D
except NameError:
    pass
gc.collect()
try:
    torch.cuda.empty_cache()
    free = torch.cuda.mem_get_info(0)[0] / 1024**3
    print(f"GPU 0 free after cleanup: {free:.1f}GB")
except RuntimeError:
    print("GPU info unavailable (CUDA driver issue). Continuing.")

## Cell 18 — Variant D+2: FoodSEM + robust URI parsing only (Idea 2 only)

Ablation: Idea 2 only (robust URI parsing, no label-aware ordering).
- **Idea 2**: `_parse_uri()` extracts suffix, validates `[A-Z]+_\d+` pattern, reconstructs `http://purl.obolibrary.org/obo/` prefix, validates against FoodOn metadata.
- **Idea 1 disabled**: `entity_labels` not set — entities processed in the original order.
Baseline for isolating the contribution of label-aware ordering (Idea 1).


In [ ]:
# Cell 18 — Variant D+2: FoodSEM + robust URI parsing only (Idea 2 only)
# entity_labels NOT set → Idea 1 disabled (no label-aware ordering).
# use_robust_uri_parsing=True → Idea 2 enabled (_parse_uri() active).

if not CUDA_AVAILABLE:
    print("SKIP: Variant D+2 requires CUDA GPU.")
    gt_metrics_D2, gliner_metrics_D2 = None, None
    gt_results_D2, gliner_results_D2 = {}, {}
else:
    from src.nel import FoodSEMNELLinker

    print("Loading Variant D+2 (FoodSEM + robust URI parsing, no label ordering)...")
    t0 = time.time()
    linker_D2 = FoodSEMNELLinker(
        local_base_model_path=LLAMA_LOCAL,
        local_adapter_path=FOODSEM_LOCAL,
        batch_size=8,
        metadata_path=METADATA_PATH,
        use_robust_uri_parsing=True,   # Idea 2 enabled
        # entity_labels not set — Idea 1 disabled (no label ordering)
    )
    linker_D2.load()
    print(f"Loaded in {time.time() - t0:.2f}s\n")

    print("--- Variant D+2 on Ground Truth entities ---")
    t0 = time.time()
    gt_results_D2 = linker_D2.link_batch(sorted(GROUND_TRUTH), context=PASSAGE)
    gt_runtime_D2 = time.time() - t0
    print_nel_table("Variant D+2 (FoodSEM + Idea 2) — GT batch", gt_results_D2, gt_runtime_D2)
    gt_metrics_D2 = nel_summary_metrics(gt_results_D2, GROUND_TRUTH)
    print(
        f"\nSummary: linked={gt_metrics_D2['linked']}, NIL={gt_metrics_D2['nil']}, "
        f"linked_rate={gt_metrics_D2['linked_rate']:.3f}, "
        f"GT_coverage={gt_metrics_D2['gt_coverage']:.3f}"
    )

    print("\n--- Variant D+2 on GLiNER-extracted entities ---")
    t0 = time.time()
    gliner_results_D2 = linker_D2.link_batch(sorted(gliner_extracted), context=PASSAGE)
    gliner_runtime_D2 = time.time() - t0
    gliner_metrics_D2 = nel_summary_metrics(gliner_results_D2, GROUND_TRUTH)
    print(
        f"Summary: linked={gliner_metrics_D2['linked']}, NIL={gliner_metrics_D2['nil']}, "
        f"GT_coverage={gliner_metrics_D2['gt_coverage']:.3f}, "
        f"runtime={gliner_runtime_D2:.2f}s"
    )

# Release GPU memory
import gc
try:
    del linker_D2
except NameError:
    pass
gc.collect()
try:
    torch.cuda.empty_cache()
    free = torch.cuda.mem_get_info(0)[0] / 1024**3
    print(f"GPU 0 free after cleanup: {free:.1f}GB")
except RuntimeError:
    print("GPU info unavailable (CUDA driver issue). Continuing.")

## Cell 19 — Variant E: FoodSEM + label-aware ordering + robust URI parsing (Ideas 1+2)

Full refinement of Variant D. Adds Idea 2 on top of Idea 1:
- **Idea 1**: food/nutrient entities sorted first within each batch (anchors FoodOn URI namespace early)
- **Idea 2**: `_parse_uri()` extracts URI suffix, validates `[A-Z]+_\d+` pattern, reconstructs with correct `http://purl.obolibrary.org/obo/` prefix, validates against FoodOn metadata


In [ ]:
# Cell 19 — Variant E: FoodSEM + label-aware ordering + robust URI parsing (Ideas 1+2)
# Loads model independently after Cell 18 released GPU memory.
# Both ideas active: entity_labels (Idea 1) + use_robust_uri_parsing (Idea 2).

if not CUDA_AVAILABLE:
    print("SKIP: Variant E requires CUDA GPU.")
    gt_metrics_E, gliner_metrics_E = None, None
    gt_results_E, gliner_results_E = {}, {}
else:
    from src.nel import FoodSEMNELLinker

    print("Loading Variant E (FoodSEM + label-aware ordering + robust URI parsing)...")
    t0 = time.time()
    linker_E = FoodSEMNELLinker(
        local_base_model_path=LLAMA_LOCAL,
        local_adapter_path=FOODSEM_LOCAL,
        batch_size=8,
        metadata_path=METADATA_PATH,
        use_robust_uri_parsing=True,   # Idea 2 enabled
    )
    linker_E.load()

    # Pass GLiNER labels to enable label-aware batch ordering (Idea 1)
    linker_E.entity_labels = {
        ent["text"]: ent["label"]
        for ent in gliner_entities_raw
    }
    print(f"Entity labels set: {len(linker_E.entity_labels)} entities")
    print("Label-aware ordering: food/nutrient entities processed first in each batch")
    print(f"Loaded in {time.time() - t0:.2f}s\n")

    print("--- Variant E on Ground Truth entities ---")
    t0 = time.time()
    gt_results_E = linker_E.link_batch(sorted(GROUND_TRUTH), context=PASSAGE)
    gt_runtime_E = time.time() - t0
    print_nel_table("Variant E (FoodSEM + Ideas 1+2) — GT batch", gt_results_E, gt_runtime_E)
    gt_metrics_E = nel_summary_metrics(gt_results_E, GROUND_TRUTH)
    print(
        f"\nSummary: linked={gt_metrics_E['linked']}, NIL={gt_metrics_E['nil']}, "
        f"linked_rate={gt_metrics_E['linked_rate']:.3f}, "
        f"GT_coverage={gt_metrics_E['gt_coverage']:.3f}"
    )
    print("NOTE: confidence=1.0 is artificial — FoodSEM generative mode has no calibrated scores.")

    print("\n--- Variant E on GLiNER-extracted entities ---")
    t0 = time.time()
    gliner_results_E = linker_E.link_batch(sorted(gliner_extracted), context=PASSAGE)
    gliner_runtime_E = time.time() - t0
    gliner_metrics_E = nel_summary_metrics(gliner_results_E, GROUND_TRUTH)
    print(
        f"Summary: linked={gliner_metrics_E['linked']}, NIL={gliner_metrics_E['nil']}, "
        f"GT_coverage={gliner_metrics_E['gt_coverage']:.3f}, "
        f"runtime={gliner_runtime_E:.2f}s"
    )

    # Release GPU memory
    import gc
    try:
        del linker_E
    except NameError:
        pass
    gc.collect()
    try:
        torch.cuda.empty_cache()
        free = torch.cuda.mem_get_info(0)[0] / 1024**3
        print(f"GPU 0 free after cleanup: {free:.1f}GB")
    except RuntimeError:
        print("GPU info unavailable (CUDA driver issue). Continuing.")

In [ ]:
# Cell 19b — Variant G: SciFoodNER foodon as black-box NER+NEL
# Results loaded from notebook_artifacts/scifoodner_results.json.
# BioBERT outputs entity spans + FoodOn URIs in a single forward pass.
# No HNSW, no external linker — entirely different architecture from Variants A-E.

if not scifoodner_available:
    print("SKIP: Run notebooks/scifoodner_evaluation.ipynb first.")
    gt_metrics_G     = None
    gliner_metrics_G = None
    gt_results_G     = {}
    gliner_results_G = {}
else:
    G = scifoodner_results["variant_G"]

    # Wrap SciFoodNER results as NELResult-compatible objects
    class _SciFoodNERResult:
        """Lightweight NELResult-compatible wrapper for SciFoodNER output."""
        def __init__(self, uri):
            self.uri             = uri
            self.canonical_label = uri.split("/")[-1] if uri else None
            self.confidence      = 1.0 if uri else 0.0
            self.method          = "scifoodner" if uri else "nil"

    # GT evaluation: for each GT entity, use SciFoodNER's URI if it found it
    gt_results_G = {}
    scifoodner_links = {k.lower(): v for k, v in G["entity_uri_pairs"].items()}
    for gt_entity in GROUND_TRUTH:
        uri = scifoodner_links.get(gt_entity.lower())
        gt_results_G[gt_entity] = _SciFoodNERResult(uri=uri)

    gt_metrics_G = nel_summary_metrics(gt_results_G, GROUND_TRUTH)

    # GLiNER pipeline: GLiNER-extracted entities linked via SciFoodNER foodon URIs
    gliner_results_G = {}
    for sf in gliner_extracted:
        uri = scifoodner_links.get(sf.lower())
        gliner_results_G[sf] = _SciFoodNERResult(uri=uri)
    gliner_metrics_G = nel_summary_metrics(gliner_results_G, GROUND_TRUTH)

    # Print summary table
    print("=" * 80)
    print("NEL: Variant G (SciFoodNER foodon — black-box NER+NEL)")
    print("=" * 80)
    print(f"{'Entity':<35} {'Method':>12} {'URI segment':<30}")
    print("-" * 80)
    for entity, res in sorted(gt_results_G.items()):
        uri_seg = res.uri.split("/")[-1] if res.uri else "NIL"
        print(f"{entity:<35} {res.method:>12} {uri_seg:<30}")

    print(f"\nSummary: linked={gt_metrics_G['linked']}, NIL={gt_metrics_G['nil']}, "
          f"GT_coverage={gt_metrics_G['gt_coverage']:.3f}")
    print(f"Runtime: {G['ner_runtime']:.2f}s (single BioBERT forward pass)")
    print(f"\nNOTE: Variant G pipeline GT coverage (own NER+NEL): "
          f"{G['pipeline_gt_coverage']:.3f}")
    print("      GLiNER pipeline GT coverage uses SciFoodNER links for GLiNER entities.")


In [ ]:
# Check Lexical linker results directly
print(f"GROUND_TRUTH size: {len(GROUND_TRUTH)}")
print(f"Lexical linked: {gt_metrics_A['linked']}")
print(f"Lexical NIL: {gt_metrics_A['nil']}")
print(f"Lexical GT coverage: {gt_metrics_A['gt_coverage']:.3f}")
print()
# Show what changed vs previous run
print("Entities that Lexical linked:")
for sf, r in sorted(gt_results_A.items()):
    if r.uri is not None:
        print(f"  {sf} → {r.uri.split('/')[-1]} ({r.canonical_label})")
print()
print("Entities that Lexical returned NIL:")
for sf, r in sorted(gt_results_A.items()):
    if r.uri is None:
        print(f"  {sf} (conf={r.confidence:.3f})")

## Cell 20 — Cross-Variant NEL Summary

GT Coverage: fraction of ground truth entities successfully linked.
Linked Rate: fraction of input entities that got a URI (not NIL).
**What to look for:** Trade-off between GT Coverage (recall) and Linked Rate (precision proxy). High Linked Rate with low GT Coverage signals many false positives.

In [ ]:
nel_summary_rows = []

if status["lexical"]:
    nel_summary_rows.append(("A — Lexical",             gt_metrics_A,             gt_runtime_A))
if status.get("sapbert"):
    nel_summary_rows.append(("B — SapBERT",             gt_metrics_B_sap,         gt_runtime_B_sap))
if status.get("biolord"):
    nel_summary_rows.append(("B — BioLORD",             gt_metrics_B_bio,         gt_runtime_B_bio))
if status.get("minilm"):
    nel_summary_rows.append(("B — MiniLM",              gt_metrics_B_mini,        gt_runtime_B_mini))
if status.get("mpnet"):
    nel_summary_rows.append(("B — MPNET",               gt_metrics_B_mpnet,       gt_runtime_B_mpnet))
if status.get("sapbert") and 'gt_metrics_B_prime' in dir() and gt_metrics_B_prime is not None:
    nel_summary_rows.append(("B' — SapBERT+CrossEnc",  gt_metrics_B_prime,       gt_runtime_B_prime))
if status.get("biolord") and 'gt_metrics_B_prime_bio' in dir() and gt_metrics_B_prime_bio is not None:
    nel_summary_rows.append(("B' — BioLORD+CrossEnc",  gt_metrics_B_prime_bio,   gt_runtime_B_prime_bio))
if status.get("minilm") and 'gt_metrics_B_prime_mini' in dir() and gt_metrics_B_prime_mini is not None:
    nel_summary_rows.append(("B' — MiniLM+CrossEnc",   gt_metrics_B_prime_mini,  gt_runtime_B_prime_mini))
if status.get("mpnet") and 'gt_metrics_B_prime_mpnet' in dir() and gt_metrics_B_prime_mpnet is not None:
    nel_summary_rows.append(("B' — MPNET+CrossEnc",    gt_metrics_B_prime_mpnet, gt_runtime_B_prime_mpnet))
if CUDA_AVAILABLE and status.get("sapbert") and 'gt_metrics_C' in dir() and gt_metrics_C is not None:
    nel_summary_rows.append(("C — HNSW+FoodSEM",       gt_metrics_C,             gt_runtime_C))
if CUDA_AVAILABLE and 'gt_metrics_D' in dir() and gt_metrics_D is not None:
    nel_summary_rows.append(("D — FoodSEM alone",      gt_metrics_D,             gt_runtime_D))
if scifoodner_available and 'gt_metrics_G' in dir() and gt_metrics_G is not None:
    nel_summary_rows.append(("G — SciFoodNER foodon",  gt_metrics_G,
                              scifoodner_results["variant_G"]["ner_runtime"]))

print(f"{'Variant':<28} {'GT Coverage':>12} {'Linked Rate':>12} {'NIL':>5} {'Runtime':>9}")
print("-" * 74)
for name, m, rt in nel_summary_rows:
    print(f"{name:<28} {m['gt_coverage']:>12.3f} {m['linked_rate']:>12.3f} "
          f"{m['nil']:>5} {rt:>8.2f}s")


## Cell 21 — NER + NEL Pipeline Summary

Combines GLiNER (NER) with each available NEL variant.
- **NER Jaccard**: quality of entity extraction vs. ground truth
- **NEL GT Coverage** (on GT entities): how many GT entities get a FoodOn URI
- **Pipeline GT Coverage** (NER → NEL): how many GT entities survive the full pipeline

**What to look for:** The pipeline GT Coverage is the key end-to-end metric. It penalises both NER misses (no entity extracted) and NEL NIL decisions.

In [ ]:
print("NER + NEL Pipeline Summary")
print(f"GLiNER NER Jaccard: {gliner_metrics['jaccard']:.3f}  "
      f"(extracted {gliner_metrics['extracted']} entities, "
      f"recall={gliner_metrics['recall']:.3f})")
print()

pipeline_rows = []

def pipeline_gt_coverage(gliner_results, ground_truth):
    gt_lower = {g.lower() for g in ground_truth}
    covered  = {e for e, r in gliner_results.items()
                if r.uri is not None and e.lower() in gt_lower}
    return len(covered) / len(ground_truth) if ground_truth else 0.0

if status["lexical"]:
    cov = pipeline_gt_coverage(gliner_results_A, GROUND_TRUTH)
    pipeline_rows.append(("A — Lexical",             gliner_metrics['jaccard'], gt_metrics_A['gt_coverage'],            cov))
if status.get("sapbert"):
    cov = pipeline_gt_coverage(gliner_results_B_sap, GROUND_TRUTH)
    pipeline_rows.append(("B — SapBERT",             gliner_metrics['jaccard'], gt_metrics_B_sap['gt_coverage'],        cov))
if status.get("biolord"):
    cov = pipeline_gt_coverage(gliner_results_B_bio, GROUND_TRUTH)
    pipeline_rows.append(("B — BioLORD",             gliner_metrics['jaccard'], gt_metrics_B_bio['gt_coverage'],        cov))
if status.get("minilm"):
    cov = pipeline_gt_coverage(gliner_results_B_mini, GROUND_TRUTH)
    pipeline_rows.append(("B — MiniLM",              gliner_metrics['jaccard'], gt_metrics_B_mini['gt_coverage'],       cov))
if status.get("mpnet"):
    cov = pipeline_gt_coverage(gliner_results_B_mpnet, GROUND_TRUTH)
    pipeline_rows.append(("B — MPNET",               gliner_metrics['jaccard'], gt_metrics_B_mpnet['gt_coverage'],      cov))
if status.get("sapbert") and 'gliner_results_B_prime' in dir() and gt_metrics_B_prime is not None:
    cov = pipeline_gt_coverage(gliner_results_B_prime, GROUND_TRUTH)
    pipeline_rows.append(("B' — SapBERT+CrossEnc",  gliner_metrics['jaccard'], gt_metrics_B_prime['gt_coverage'],      cov))
if status.get("biolord") and 'gliner_results_B_prime_bio' in dir() and gt_metrics_B_prime_bio is not None:
    cov = pipeline_gt_coverage(gliner_results_B_prime_bio, GROUND_TRUTH)
    pipeline_rows.append(("B' — BioLORD+CrossEnc",  gliner_metrics['jaccard'], gt_metrics_B_prime_bio['gt_coverage'],  cov))
if status.get("minilm") and 'gliner_results_B_prime_mini' in dir() and gt_metrics_B_prime_mini is not None:
    cov = pipeline_gt_coverage(gliner_results_B_prime_mini, GROUND_TRUTH)
    pipeline_rows.append(("B' — MiniLM+CrossEnc",   gliner_metrics['jaccard'], gt_metrics_B_prime_mini['gt_coverage'], cov))
if status.get("mpnet") and 'gliner_results_B_prime_mpnet' in dir() and gt_metrics_B_prime_mpnet is not None:
    cov = pipeline_gt_coverage(gliner_results_B_prime_mpnet, GROUND_TRUTH)
    pipeline_rows.append(("B' — MPNET+CrossEnc",    gliner_metrics['jaccard'], gt_metrics_B_prime_mpnet['gt_coverage'], cov))
if CUDA_AVAILABLE and status.get("sapbert") and 'gliner_results_C' in dir() and gt_metrics_C is not None:
    cov = pipeline_gt_coverage(gliner_results_C, GROUND_TRUTH)
    pipeline_rows.append(("C — HNSW+FoodSEM",       gliner_metrics['jaccard'], gt_metrics_C['gt_coverage'],            cov))
if CUDA_AVAILABLE and 'gliner_results_D' in dir() and gt_metrics_D is not None:
    cov = pipeline_gt_coverage(gliner_results_D, GROUND_TRUTH)
    pipeline_rows.append(("D — FoodSEM alone",      gliner_metrics['jaccard'], gt_metrics_D['gt_coverage'],            cov))
if scifoodner_available and 'gt_metrics_G' in dir() and gt_metrics_G is not None:
    cov_gliner = pipeline_gt_coverage(gliner_results_G, GROUND_TRUTH)
    pipeline_rows.append(("G — GLiNER+SciFoodNER",  gliner_metrics['jaccard'], gt_metrics_G['gt_coverage'],            cov_gliner))
    pipeline_rows.append(("G — SciFoodNER e2e",     None,                      gt_metrics_G['gt_coverage'],
                          scifoodner_results["variant_G"]["pipeline_gt_coverage"]))

print(f"{'NEL Variant':<28} {'NER Jaccard':>12} {'NEL GT Cov':>11} {'Pipeline GT Cov':>16}")
print("-" * 72)
for name, ner_j, nel_cov, pipe_cov in pipeline_rows:
    ner_j_str = f"{ner_j:>12.3f}" if ner_j is not None else f"{'(own NER)':>12}"
    print(f"{name:<28} {ner_j_str} {nel_cov:>11.3f} {pipe_cov:>16.3f}")

print()
print("Interpretation:")
print("  Pipeline GT Coverage = fraction of GT entities correctly extracted AND linked.")
print("  Higher is better. Gap vs NER recall = NEL-induced loss.")
